# Star Wars Movies — Full Analytics Pipeline
**File:** Dataset_Notebook_1.ipynb  
**Stack:** DuckDB → Neo4j → GDS → scikit-learn → Qdrant → Streamlit  
**Business Question:** Which actors/directors should a streaming platform prioritise for VFX-heavy endorsement?

In [1]:
import os, zipfile

print("Current directory:", os.getcwd())
print()

print("Files in current folder:")
for f in sorted(os.listdir(".")):
    size = os.path.getsize(f) / (1024*1024)
    print(f"  {f}  ({size:.1f} MB)")

print()

if os.path.exists("data"):
    print("Files in data/ folder:")
    for f in sorted(os.listdir("data")):
        size = os.path.getsize(os.path.join("data", f)) / (1024*1024)
        print(f"  data/{f}  ({size:.1f} MB)")
else:
    print("No data/ folder found")

Current directory: c:\Users\evame\OneDrive\Documents\GitHub\big_data_capstone_project\data

Files in current folder:
  Dataset_Notebook_1.ipynb  (0.0 MB)
  Dataset_Notebook_2.ipynb  (0.1 MB)
  Dataset_Notebook_3.ipynb  (0.1 MB)
  Dataset_Notebook_4.ipynb  (0.0 MB)
  Dataset_Notebook_5.ipynb  (0.0 MB)
  VFX_MCU_DC_Analysis_professional.ipynb  (1.8 MB)
  neo4j_directed_by_1.csv  (3.7 MB)
  neo4j_directors_1.csv  (1.6 MB)
  neo4j_movies_1.csv  (10.6 MB)
  starwars_transformers.parquet  (0.1 MB)
  vfx_movies_1.parquet  (61.4 MB)

No data/ folder found


In [2]:
import pandas as pd
import numpy as np
import duckdb
import zipfile
import os
import json
import warnings
from collections import defaultdict

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 80)

print("✅ All imports ready")

✅ All imports ready


---
## SECTION 1 — ETL: Extract Star Wars Films with DuckDB
---

In [3]:
import os

print("Current directory:", os.getcwd())
print()

print("Everything in current folder:")
for f in sorted(os.listdir(".")):
    print(" ", f)

print()

if os.path.exists("data"):
    print("Everything in data/ folder:")
    for f in sorted(os.listdir("data")):
        full = os.path.join("data", f)
        size = os.path.getsize(full) / (1024*1024)
        print(f"  {f}  →  {size:.0f} MB")

Current directory: c:\Users\evame\OneDrive\Documents\GitHub\big_data_capstone_project\data

Everything in current folder:
  Dataset_Notebook_1.ipynb
  Dataset_Notebook_2.ipynb
  Dataset_Notebook_3.ipynb
  Dataset_Notebook_4.ipynb
  Dataset_Notebook_5.ipynb
  VFX_MCU_DC_Analysis_professional.ipynb
  neo4j_directed_by_1.csv
  neo4j_directors_1.csv
  neo4j_movies_1.csv
  starwars_transformers.parquet
  vfx_movies_1.parquet



In [4]:
ZIP_PATH     = r"C:\Users\evame\Downloads\archive\TMDB_all_movies.zip"   # ← your zip file
OUT_PARQUET  = "data/starwars_films.parquet"

# Step 1: Read CSV from inside the zip
print("Reading zip file...")
with zipfile.ZipFile(ZIP_PATH) as z:
    csv_name = [f for f in z.namelist() if f.endswith(".csv")][0]
    print(f"   Found inside zip: {csv_name}")
    with z.open(csv_name) as f:
        df_all = pd.read_csv(f, low_memory=False)

print(f"   Total rows in full dataset: {len(df_all):,}")
print(f"   Columns: {df_all.columns.tolist()}")

Reading zip file...
   Found inside zip: TMDB_all_movies.csv
   Total rows in full dataset: 1,190,921
   Columns: ['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'budget', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'cast', 'director', 'director_of_photography', 'writers', 'producers', 'music_composer', 'imdb_rating', 'imdb_votes', 'poster_path']


In [5]:
# Filter Star Wars + Transformers with DuckDB

con = duckdb.connect()
con.register("movies", df_all)

vfx = con.execute("""
    SELECT *,
        CASE
            WHEN lower(title) LIKE '%star wars%'
              OR lower(original_title) LIKE '%star wars%'
            THEN 'Star Wars'
            ELSE 'Transformers'
        END AS franchise
    FROM movies
    WHERE (
        -- Star Wars
        (
            (lower(title) LIKE '%star wars%'
             OR lower(original_title) LIKE '%star wars%')
            AND lower(title) NOT LIKE '%wccw%'
            AND lower(title) NOT LIKE '%wrestling%'
            AND lower(title) NOT LIKE '%lone star%'
            AND lower(title) NOT LIKE '%doraemon%'
        )
        OR
        -- Transformers
        lower(title) LIKE '%transformers%'
    )
    AND revenue > 1000000
    AND status = 'Released'
    ORDER BY release_date
""").df()

print(f"Total VFX films: {len(vfx)}")
print(f"Star Wars:    {len(vfx[vfx['franchise']=='Star Wars'])}")
print(f"Transformers: {len(vfx[vfx['franchise']=='Transformers'])}")
print()
print(vfx[["franchise","title","release_date",
           "revenue","imdb_rating","director"]].to_string(index=False))

Total VFX films: 18
Star Wars:    10
Transformers: 8

   franchise                                        title release_date          revenue  imdb_rating         director
   Star Wars                                    Star Wars   1977-05-25   775,398,007.00         8.60     George Lucas
Transformers                  The Transformers: The Movie   1986-08-08     5,860,601.00         7.20      Nelson Shin
   Star Wars    Star Wars: Episode I - The Phantom Menace   1999-05-19   924,317,558.00         6.50     George Lucas
   Star Wars Star Wars: Episode II - Attack of the Clones   2002-05-15   649,398,328.00         6.60     George Lucas
   Star Wars Star Wars: Episode III - Revenge of the Sith   2005-05-17   850,000,000.00         7.70     George Lucas
Transformers                                 Transformers   2007-06-27   709,709,780.00         7.10      Michael Bay
   Star Wars                    Star Wars: The Clone Wars   2008-08-05    68,282,844.00         6.00      Dave Filoni
Tr

In [6]:
# Clean + Feature Engineering

df = vfx.copy()

# Force numeric columns
for col in ["revenue","budget","vote_average","imdb_rating",
            "imdb_votes","popularity","runtime"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Release year
df["release_year"] = pd.to_datetime(
    df["release_date"], errors="coerce"
).dt.year.astype("Int64")

# Financial features
df["profit"]        = df["revenue"] - df["budget"]
df["roi"]           = df.apply(
    lambda r: round(r["profit"]/r["budget"], 2) if r["budget"] > 0 else 0,
    axis=1
)
df["profit_margin"] = df.apply(
    lambda r: round(r["profit"]/r["revenue"]*100, 1) if r["revenue"] > 0 else 0,
    axis=1
)

# Short title for charts
df["short_title"] = (
    df["title"]
    .str.replace("Star Wars: Episode", "Ep",        regex=False)
    .str.replace("Star Wars: The",     "The",        regex=False)
    .str.replace("Star Wars: ",        "",           regex=False)
    .str.replace(": A Star Wars Story","",           regex=False)
    .str.replace("Star Wars",          "A New Hope", regex=False)
    .str.replace("Transformers: ",     "TF: ",       regex=False)
)

# Era for Star Wars, Phase for Transformers
def get_era(row):
    t   = str(row["title"]).lower()
    yr  = row.get("release_year", 0)
    frn = row["franchise"]

    if frn == "Star Wars":
        if any(x in t for x in ["episode i","phantom","episode ii",
                                  "clones","episode iii","sith"]):
            return "SW Prequel"
        if any(x in t for x in ["episode iv","new hope"]) or yr == 1977:
            return "SW Original"
        if any(x in t for x in ["force awakens","last jedi","rise of skywalker"]):
            return "SW Sequel"
        if any(x in t for x in ["rogue one","solo"]):
            return "SW Anthology"
        return "SW Other"
    else:
        if yr <= 1986:  return "TF Classic"
        if yr <= 2011:  return "TF Bay Era 1"
        if yr <= 2017:  return "TF Bay Era 2"
        return "TF New Era"

df["era"] = df.apply(get_era, axis=1)

# Cast as list (top 20 billed)
df["cast_list"] = df["cast"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c.strip()][:20]
)

# Text for embedding = title + overview + tagline
df["embed_text"] = (
    df["title"].fillna("") + ". " +
    df["overview"].fillna("") + " " +
    df["tagline"].fillna("")
).str.strip()

# Save parquet
df.to_parquet("starwars_transformers.parquet", index=False)

print(f" Saved → starwars_transformers.parquet")
print(f"   Total films : {len(df)}")
print(f"   Star Wars   : {len(df[df['franchise']=='Star Wars'])}")
print(f"   Transformers: {len(df[df['franchise']=='Transformers'])}")
print()
print(df[["short_title","franchise","era","release_year",
          "revenue","budget","roi","imdb_rating"]].to_string(index=False))

 Saved → starwars_transformers.parquet
   Total films : 18
   Star Wars   : 10
   Transformers: 8

                 short_title    franchise          era  release_year          revenue         budget   roi  imdb_rating
                  A New Hope    Star Wars  SW Original          1977   775,398,007.00  11,000,000.00 69.49         8.60
           The TF: The Movie Transformers   TF Classic          1986     5,860,601.00   6,000,000.00 -0.02         7.20
   Ep I - The Phantom Menace    Star Wars   SW Prequel          1999   924,317,558.00 115,000,000.00  7.04         6.50
Ep II - Attack of the Clones    Star Wars   SW Prequel          2002   649,398,328.00 120,000,000.00  4.41         6.60
Ep III - Revenge of the Sith    Star Wars   SW Prequel          2005   850,000,000.00 113,000,000.00  6.52         7.70
                Transformers Transformers TF Bay Era 1          2007   709,709,780.00 150,000,000.00  3.73         7.10
              The Clone Wars    Star Wars     SW Other       

In [7]:
import subprocess
subprocess.run(["pip", "install", "nbformat", "--upgrade", "-q"], check=True)
print("Done")

Done


In [8]:
#Overview Charts

import plotly.express as px
import plotly.graph_objects as go

FRANCHISE_COLOR = {
    "Star Wars":   "#FFD700",
    "Transformers":"#4A90D9",
}

ERA_COLOR = {
    "SW Original":   "#F5A623",
    "SW Prequel":    "#4A90D9",
    "SW Sequel":     "#7ED321",
    "SW Anthology":  "#D0021B",
    "SW Other":      "#9B59B6",
    "TF Classic":    "#1ABC9C",
    "TF Bay Era 1":  "#2C3E50",
    "TF Bay Era 2":  "#8E44AD",
    "TF New Era":    "#E74C3C",
}

# Chart 1 — Revenue by film
fig1 = px.bar(
    df.sort_values("revenue"),
    x="revenue", y="short_title",
    orientation="h", color="franchise",
    color_discrete_map=FRANCHISE_COLOR,
    title="Box Office Revenue — Star Wars vs Transformers",
    labels={"revenue":"Revenue (USD)","short_title":""},
    text=df.sort_values("revenue")["revenue"]
           .apply(lambda x: f"${x/1e9:.2f}B")
)
fig1.update_traces(textposition="outside")
fig1.update_layout(height=600)
fig1.show()

# Chart 2 — Revenue timeline
fig2 = px.scatter(
    df.sort_values("release_year"),
    x="release_year", y="revenue",
    color="franchise", size="popularity",
    hover_name="short_title",
    color_discrete_map=FRANCHISE_COLOR,
    title="Revenue Timeline (bubble = popularity)",
    labels={"release_year":"Year","revenue":"Revenue (USD)"}
)
fig2.show()

# Chart 3 — IMDB comparison
fig3 = px.bar(
    df.sort_values("imdb_rating", ascending=False),
    x="short_title", y="imdb_rating",
    color="franchise",
    color_discrete_map=FRANCHISE_COLOR,
    title="IMDB Ratings Comparison",
    labels={"imdb_rating":"IMDB Rating","short_title":""}
)
fig3.update_layout(xaxis_tickangle=-35)
fig3.show()

# Chart 4 — Budget vs Revenue
fig4 = px.scatter(
    df, x="budget", y="revenue",
    color="franchise", hover_name="short_title",
    size="imdb_votes",
    color_discrete_map=FRANCHISE_COLOR,
    title="Budget vs Revenue (bubble = IMDB votes)",
    labels={"budget":"Budget (USD)","revenue":"Revenue (USD)"}
)
max_v = max(df["budget"].max(), df["revenue"].max()) * 1.1
fig4.add_shape(type="line", x0=0, y0=0, x1=max_v, y1=max_v,
               line=dict(dash="dash", color="gray"))
fig4.add_annotation(x=max_v*0.7, y=max_v*0.6,
                    text="Break-even", showarrow=False,
                    font=dict(size=10, color="gray"))
fig4.show()

# Chart 5 — ROI by film
fig5 = px.bar(
    df.sort_values("roi", ascending=False),
    x="short_title", y="roi",
    color="franchise",
    color_discrete_map=FRANCHISE_COLOR,
    title="Return on Investment by Film",
    labels={"roi":"ROI (×)","short_title":""}
)
fig5.update_layout(xaxis_tickangle=-35)
fig5.show()

print("Overview charts done")

Overview charts done


---
## SECTION 2 — Machine Learning
---

In [10]:
# Build ML feature matrix from our 18 films

from sklearn.preprocessing import LabelEncoder

ml_df = df.copy()

# Encode categorical columns
le_franchise = LabelEncoder()
le_era       = LabelEncoder()

ml_df["franchise_enc"] = le_franchise.fit_transform(ml_df["franchise"])
ml_df["era_enc"]       = le_era.fit_transform(ml_df["era"])

print("Franchise encoding:", dict(zip(le_franchise.classes_,
                                       le_franchise.transform(le_franchise.classes_))))
print("Era encoding:",       dict(zip(le_era.classes_,
                                       le_era.transform(le_era.classes_))))

# Target variable — is this film a blockbuster?
median_rev            = ml_df["revenue"].median()
ml_df["is_blockbuster"] = (ml_df["revenue"] > median_rev).astype(int)

print(f"\nMedian revenue    : ${median_rev:,.0f}")
print(f"Blockbusters  (1) : {ml_df['is_blockbuster'].sum()}")
print(f"Non-blockbuster(0): {(ml_df['is_blockbuster']==0).sum()}")
print()

# Use cast count as crew size proxy
ml_df["crew_size"] = ml_df["cast_list"].apply(len)

# Feature columns
FEATURE_COLS = [
    "budget",
    "runtime",
    "vote_average",
    "popularity",
    "imdb_rating",
    "franchise_enc",
    "era_enc",
    "crew_size",
]

# Make sure all features are numeric
for col in FEATURE_COLS:
    ml_df[col] = pd.to_numeric(ml_df[col], errors="coerce").fillna(0)

print("=== Feature Matrix ===")
print(ml_df[["short_title","franchise"] +
            FEATURE_COLS +
            ["revenue","is_blockbuster"]].to_string(index=False))

Franchise encoding: {'Star Wars': np.int64(0), 'Transformers': np.int64(1)}
Era encoding: {'SW Anthology': np.int64(0), 'SW Original': np.int64(1), 'SW Other': np.int64(2), 'SW Prequel': np.int64(3), 'SW Sequel': np.int64(4), 'TF Bay Era 1': np.int64(5), 'TF Bay Era 2': np.int64(6), 'TF Classic': np.int64(7), 'TF New Era': np.int64(8)}

Median revenue    : $805,851,087
Blockbusters  (1) : 9
Non-blockbuster(0): 9

=== Feature Matrix ===
                 short_title    franchise         budget  runtime  vote_average  popularity  imdb_rating  franchise_enc  era_enc  crew_size          revenue  is_blockbuster
                  A New Hope    Star Wars  11,000,000.00   121.00          8.20       25.71         8.60              0        1         20   775,398,007.00               0
           The TF: The Movie Transformers   6,000,000.00    85.00          7.12        4.25         7.20              1        7         20     5,860,601.00               0
   Ep I - The Phantom Menace    Star Wars

In [14]:
# Train 3 models using Leave-One-Out CV
# LOO is correct here because we only have 18 films

from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics         import classification_report
import warnings
warnings.filterwarnings("ignore")

X   = ml_df[FEATURE_COLS].values
y   = ml_df["is_blockbuster"].values
loo = LeaveOneOut()

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    RandomForestClassifier(
                       n_estimators=100,
                       max_depth=3,
                       random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    GradientBoostingClassifier(
                       n_estimators=50,
                       max_depth=2,
                       random_state=42))
    ]),
}

print("=== Model Comparison — Leave-One-Out CV ===")
print("(LOO used because dataset has only 18 films)\n")

best_name, best_pipe, best_score = None, None, 0

for name, pipe in models.items():
    cv_scores = cross_val_score(pipe, X, y, cv=loo, scoring="accuracy")
    pipe.fit(X, y)
    y_pred = pipe.predict(X)

    print(f"── {name}")
    print(f"   LOO Accuracy : {cv_scores.mean():.3f} "
          f"({int(cv_scores.sum())}/{len(cv_scores)} correct)")
    print(classification_report (
        y, y_pred,
        target_names=["Non-Blockbuster","Blockbuster"],
    
    ))

    if cv_scores.mean() > best_score:
        best_score = cv_scores.mean()
        best_name  = name
        best_pipe  = pipe

print(f"Best model : {best_name}")
print(f"   LOO accuracy: {best_score:.3f}")

=== Model Comparison — Leave-One-Out CV ===
(LOO used because dataset has only 18 films)

── Logistic Regression
   LOO Accuracy : 0.500 (9/18 correct)
                 precision    recall  f1-score   support

Non-Blockbuster       0.86      0.67      0.75         9
    Blockbuster       0.73      0.89      0.80         9

       accuracy                           0.78        18
      macro avg       0.79      0.78      0.78        18
   weighted avg       0.79      0.78      0.77        18

── Random Forest
   LOO Accuracy : 0.500 (9/18 correct)
                 precision    recall  f1-score   support

Non-Blockbuster       1.00      1.00      1.00         9
    Blockbuster       1.00      1.00      1.00         9

       accuracy                           1.00        18
      macro avg       1.00      1.00      1.00        18
   weighted avg       1.00      1.00      1.00        18

── Gradient Boosting
   LOO Accuracy : 0.556 (10/18 correct)
                 precision    recall  f1-

In [15]:
# Feature importance and predictions on all 18 films

from sklearn.inspection import permutation_importance

# Permutation importance
perm = permutation_importance(
    best_pipe, X, y,
    n_repeats=20,
    random_state=42
)

feat_imp = pd.DataFrame({
    "feature":    FEATURE_COLS,
    "importance": perm.importances_mean,
}).sort_values("importance", ascending=False)

print(f"=== Feature Importances — {best_name} ===")
print(feat_imp.to_string(index=False))

# Chart 1 — Feature importance
fig1 = px.bar(
    feat_imp.sort_values("importance"),
    x="importance", y="feature",
    orientation="h",
    color="importance",
    color_continuous_scale="Blues",
    title=f"Feature Importance — {best_name}",
    labels={"importance":"Importance","feature":""}
)
fig1.show()

# Predict all 18 films
ml_df["hit_probability"] = best_pipe.predict_proba(X)[:, 1]
ml_df["predicted_hit"]   = best_pipe.predict(X)

# Chart 2 — Hit probability per film
fig2 = px.bar(
    ml_df.sort_values("hit_probability"),
    x="hit_probability", y="short_title",
    orientation="h", color="franchise",
    color_discrete_map=FRANCHISE_COLOR,
    title="Predicted Blockbuster Probability — All 18 Films",
    labels={"hit_probability":"P(Blockbuster)","short_title":""},
    text=ml_df.sort_values("hit_probability")["hit_probability"]
              .apply(lambda x: f"{x:.0%}")
)
fig2.update_traces(textposition="outside")
fig2.update_layout(height=600)
fig2.show()

# Chart 3 — Actual revenue vs predicted probability
fig3 = px.scatter(
    ml_df,
    x="revenue", y="hit_probability",
    color="franchise", hover_name="short_title",
    size="budget",
    color_discrete_map=FRANCHISE_COLOR,
    title="Actual Revenue vs Predicted Probability (bubble = budget)",
    labels={
        "revenue":         "Actual Revenue (USD)",
        "hit_probability": "P(Blockbuster)"
    }
)
fig3.add_hline(y=0.5, line_dash="dash", line_color="gray",
               annotation_text="Decision boundary")
fig3.show()

# Chart 4 — Correct vs wrong predictions
ml_df["prediction_result"] = ml_df.apply(
    lambda r: "Correct" if r["predicted_hit"] == r["is_blockbuster"]
              else "Wrong", axis=1
)
fig4 = px.bar(
    ml_df.sort_values("revenue"),
    x="revenue", y="short_title",
    orientation="h",
    color="prediction_result",
    color_discrete_map={"Correct":"#27AE60","Wrong":"#E74C3C"},
    hover_data=["hit_probability","is_blockbuster"],
    title="Prediction Results — Correct vs Wrong",
    labels={"revenue":"Revenue (USD)","short_title":""}
)
fig4.show()

# Save predictions
ml_df[[
    "title","short_title","franchise","release_year","era",
    "revenue","budget","roi","imdb_rating",
    "hit_probability","predicted_hit","is_blockbuster","prediction_result"
]].to_csv("ml_predictions.csv", index=False)

print("\n✅ Saved → ml_predictions.csv")
print()
print(ml_df[[
    "short_title","franchise","revenue",
    "hit_probability","predicted_hit",
    "is_blockbuster","prediction_result"
]].sort_values("hit_probability", ascending=False).to_string(index=False))

=== Feature Importances — Gradient Boosting ===
      feature  importance
      runtime        0.21
       budget        0.19
   popularity        0.18
  imdb_rating        0.07
 vote_average        0.00
franchise_enc        0.00
      era_enc        0.00
    crew_size        0.00



✅ Saved → ml_predictions.csv

                 short_title    franchise          revenue  hit_probability  predicted_hit  is_blockbuster prediction_result
       The Rise of Skywalker    Star Wars 1,074,144,248.00             0.95              1               1           Correct
               The Last Jedi    Star Wars 1,334,407,706.00             0.95              1               1           Correct
           The Force Awakens    Star Wars 2,068,223,624.00             0.95              1               1           Correct
   Ep I - The Phantom Menace    Star Wars   924,317,558.00             0.95              1               1           Correct
Ep III - Revenge of the Sith    Star Wars   850,000,000.00             0.95              1               1           Correct
       TF: Age of Extinction Transformers 1,105,261,713.00             0.95              1               1           Correct
        TF: Dark of the Moon Transformers 1,123,794,079.00             0.95              1    

---
#SECTION 3 — Semantic Search: Embeddings + Qdrant
---

In [16]:
# Generate sentence embeddings for all 18 films
# Uses title + overview + tagline as input text
# Model downloads ~90MB on first run — wait for it

from sentence_transformers import SentenceTransformer
import numpy as np
import time

print("Loading model...")
print("First run downloads ~90MB — this takes 1-2 minutes\n")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Model loaded\n")

# Show what text we are embedding
print("=== Texts being embedded ===")
for i, row in df.iterrows():
    print(f"\n[{row['franchise']}] {row['short_title']}")
    print(f"  {row['embed_text'][:100]}...")

# Generate embeddings
print("\nGenerating embeddings...")
t0 = time.time()

embeddings = model.encode(
    df["embed_text"].tolist(),
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"\nDone in {time.time()-t0:.1f}s")
print(f"   Shape : {embeddings.shape}")
print(f"   Dims  : {embeddings.shape[1]} per film")
print(f"   Films : {embeddings.shape[0]}")

# Save
np.save("sw_tf_embeddings.npy", embeddings)
df[["title","short_title","franchise","era","embed_text"]]\
    .to_csv("embedding_index.csv", index=False)

print("\nsw_tf_embeddings.npy saved")
print("embedding_index.csv saved")

Loading model...
First run downloads ~90MB — this takes 1-2 minutes



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4364.17it/s]


Model loaded

=== Texts being embedded ===

[Star Wars] A New Hope
  Star Wars. Princess Leia is captured and held hostage by the evil Imperial forces in their effort to...

[Transformers] The TF: The Movie
  The Transformers: The Movie. The Autobots must stop a colossal planet-consuming robot who goes after...

[Star Wars] Ep I - The Phantom Menace
  Star Wars: Episode I - The Phantom Menace. Anakin Skywalker, a young slave strong with the Force, is...

[Star Wars] Ep II - Attack of the Clones
  Star Wars: Episode II - Attack of the Clones. Following an assassination attempt on Senator Padmé Am...

[Star Wars] Ep III - Revenge of the Sith
  Star Wars: Episode III - Revenge of the Sith. When the sinister Sith unveil a thousand-year-old plot...

[Transformers] Transformers
  Transformers. Young teenager Sam Witwicky becomes involved in the ancient struggle between two extra...

[Star Wars] The Clone Wars
  Star Wars: The Clone Wars. As the Clone Wars sweep through the galaxy, the heroic

Batches: 100%|██████████| 3/3 [00:00<00:00,  7.37it/s]


Done in 0.4s
   Shape : (18, 384)
   Dims  : 384 per film
   Films : 18

sw_tf_embeddings.npy saved
embedding_index.csv saved


In [17]:
# Semantic search using local cosine similarity
# Works without Docker / Qdrant

def semantic_search(query, top_k=5, franchise_filter=None):
    """
    Find the most similar films to a text query.
    franchise_filter: 'Star Wars' or 'Transformers' to restrict
    """
    # Encode the query
    query_vec = model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    # Cosine similarity (embeddings are normalized so dot product = cosine)
    scores = embeddings @ query_vec

    # Apply franchise filter
    if franchise_filter:
        mask   = df["franchise"] == franchise_filter
        scores = scores * mask.values

    # Get top results
    top_idx = scores.argsort()[::-1][:top_k]

    print(f"\n{'─'*60}")
    print(f"Query : '{query}'"
          + (f"  [{franchise_filter}]" if franchise_filter else ""))
    print(f"{'─'*60}")

    for idx in top_idx:
        row = df.iloc[idx]
        print(f"  [{scores[idx]:.3f}]  {row['short_title']:<35}"
              f"  {row['franchise']:<14}  {row['era']}")
        print(f"           {str(row.get('overview',''))[:80]}...")
    print()


# Test with 8 different queries
queries = [
    # theme queries
    ("machines and robots transforming and destroying cities",  None),
    ("hope and rebellion against a tyrannical empire",          None),
    ("chosen one destined to bring balance to the universe",    None),
    ("father and son conflict good vs evil",                    None),
    # franchise specific
    ("alien robots disguised as vehicles on earth",             "Transformers"),
    ("Jedi knights and the Force",                              "Star Wars"),
    # cross franchise
    ("hero sacrifices everything to save the world",            None),
    ("sequel with bigger budget but lower audience ratings",    None),
]

print("=== Semantic Search Results ===")
for query, franchise_f in queries:
    semantic_search(query, top_k=3, franchise_filter=franchise_f)

=== Semantic Search Results ===

────────────────────────────────────────────────────────────
Query : 'machines and robots transforming and destroying cities'
────────────────────────────────────────────────────────────
  [0.448]  The TF: The Movie                    Transformers    TF Classic
           The Autobots must stop a colossal planet-consuming robot who goes after the Auto...
  [0.444]  Transformers                         Transformers    TF Bay Era 1
           Young teenager Sam Witwicky becomes involved in the ancient struggle between two...
  [0.443]  TF: Age of Extinction                Transformers    TF Bay Era 2
           As humanity picks up the pieces after the battle of Chicago, a shadowy group rev...


────────────────────────────────────────────────────────────
Query : 'hope and rebellion against a tyrannical empire'
────────────────────────────────────────────────────────────
  [0.521]  Rogue One                            Star Wars       SW Anthology
        

In [19]:
# Visualise embeddings in 2D using PCA
# Shows how similar the films are to each other

from sklearn.decomposition import PCA

# Reduce 384 dimensions → 2 dimensions
pca  = PCA(n_components=2, random_state=42)
emb2d = pca.fit_transform(embeddings)

emb_df = pd.DataFrame({
    "x":           emb2d[:, 0],
    "y":           emb2d[:, 1],
    "title":       df["short_title"],
    "franchise":   df["franchise"],
    "era":         df["era"],
    "revenue":     df["revenue"],
    "imdb_rating": df["imdb_rating"],
})

print(f"PCA variance explained: "
      f"{pca.explained_variance_ratio_.sum()*100:.1f}%")

# Chart — 2D embedding space
fig = px.scatter(
    emb_df,
    x="x", y="y",
    color="franchise",
    size="revenue",
    hover_name="title",
    hover_data=["era","imdb_rating"],
    color_discrete_map=FRANCHISE_COLOR,
    text="title",
    title="Film Embedding Space — PCA 2D "
          "(similar films cluster together)",
    labels={"x":"PCA Component 1","y":"PCA Component 2"}
)
fig.update_traces(textposition="top center", textfont_size=9)
fig.update_layout(height=550)
fig.show()

# Chart — colour by era instead
fig2 = px.scatter(
    emb_df,
    x="x", y="y",
    color="era",
    size="revenue",
    hover_name="title",
    text="title",
    title="Embedding Space coloured by Era",
    labels={"x":"PCA Component 1","y":"PCA Component 2"}
)
fig2.update_traces(textposition="top center", textfont_size=9)
fig2.update_layout(height=550)
fig2.show()



PCA variance explained: 40.4%


In [20]:
# Show which films are most similar to each other

# Cosine similarity matrix (all pairs)
sim_matrix = embeddings @ embeddings.T

sim_df = pd.DataFrame(
    sim_matrix,
    index=df["short_title"].tolist(),
    columns=df["short_title"].tolist()
)

# Heatmap
fig = px.imshow(
    sim_matrix,
    x=df["short_title"].tolist(),
    y=df["short_title"].tolist(),
    color_continuous_scale="Blues",
    title="Film Similarity Matrix "
          "(darker = more similar plot/theme)",
    zmin=0, zmax=1
)
fig.update_layout(height=600)
fig.update_xaxes(tickangle=-45, tickfont_size=9)
fig.update_yaxes(tickfont_size=9)
fig.show()

# Print most similar pairs
print("=== Most Similar Film Pairs ===\n")
pairs = []
for i in range(len(df)):
    for j in range(i+1, len(df)):
        pairs.append({
            "film_a":    df.iloc[i]["short_title"],
            "film_b":    df.iloc[j]["short_title"],
            "franchise_a": df.iloc[i]["franchise"],
            "franchise_b": df.iloc[j]["franchise"],
            "similarity": round(float(sim_matrix[i,j]), 3)
        })

pairs_df = pd.DataFrame(pairs).sort_values(
    "similarity", ascending=False
)

print("Top 10 most similar pairs:")
print(pairs_df.head(10).to_string(index=False))
print()
print("Most similar CROSS-franchise pairs:")
cross = pairs_df[
    pairs_df["franchise_a"] != pairs_df["franchise_b"]
].head(5)
print(cross.to_string(index=False))

pairs_df.to_csv("film_similarity.csv", index=False)
print("\nSaved → film_similarity.csv")

=== Most Similar Film Pairs ===

Top 10 most similar pairs:
                   film_a                       film_b  franchise_a  franchise_b  similarity
             Transformers    TF: Revenge of the Fallen Transformers Transformers        0.82
            The Last Jedi        The Rise of Skywalker    Star Wars    Star Wars        0.75
    TF: Age of Extinction          TF: The Last Knight Transformers Transformers        0.73
Ep I - The Phantom Menace Ep III - Revenge of the Sith    Star Wars    Star Wars        0.73
        The TF: The Movie    TF: Revenge of the Fallen Transformers Transformers        0.71
        The TF: The Movie                 Transformers Transformers Transformers        0.70
    TF: Age of Extinction       TF: Rise of the Beasts Transformers Transformers        0.69
     TF: Dark of the Moon          TF: The Last Knight Transformers Transformers        0.69
        The TF: The Movie         TF: Dark of the Moon Transformers Transformers        0.69
        Th